In [0]:

from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
data = [
    ("Alice", 25, "New York"),
    ("Bob", 30, "San Francisco"),
    ("Charlie", 35, "Chicago"),
    ("David", 22, "New York"),
    ("Emma", 28, "San Francisco"),
    ("Frank", 40, "Chicago"),
    ("Grace", 27, "New York"),
    ("Henry", 33, "San Francisco"),
    ("Isabella", 24, "Chicago"),
    ("Jack", 31, "New York"),
    ("Karen", 29, "San Francisco"),
    ("Liam", 36, "Chicago"),
    ("Mia", 26, "New York"),
    ("Noah", 38, "San Francisco"),
    ("Olivia", 23, "Chicago"),
    ("Peter", 41, "New York"),
    ("Quinn", 32, "San Francisco"),
    ("Ryan", 37, "Chicago"),
    ("Sophia", 21, "New York"),
    ("Thomas", 34, "San Francisco"),
    ("Uma", 39, "Chicago"),
    ("Victor", 27, "New York"),
    ("William", 42, "San Francisco"),
    ("Xavier", 30, "Chicago"),
    ("Yara", 25, "New York"),
    ("Zach", 29, "San Francisco"),
    ("Aaron", 35, "Chicago"),
    ("Bella", 24, "New York"),
    ("Caleb", 31, "San Francisco"),
    ("Diana", 33, "Chicago"),
    ("Ethan", 26, "New York"),
    ("Fiona", 28, "San Francisco"),
    ("George", 40, "Chicago"),
    ("Hannah", 22, "New York"),
    ("Ian", 37, "San Francisco"),
    ("Julia", 34, "Chicago"),
    ("Kevin", 23, "New York"),
    ("Laura", 36, "San Francisco"),
    ("Michael", 41, "Chicago"),
    ("Nina", 27, "New York"),
    ("Oscar", 32, "San Francisco"),
    ("Paula", 38, "Chicago"),
    ("Richard", 24, "New York"),
    ("Sarah", 30, "San Francisco"),
    ("Tony", 35, "Chicago"),
    ("Ursula", 29, "New York"),
    ("Vincent", 33, "San Francisco"),
    ("Wendy", 26, "Chicago"),
    ("Brian", 39, "New York"),
    ("Catherine", 31, "San Francisco"),
]

schema = StructType([
    StructField("name", StringType(), True),
    StructField("age", IntegerType(), True),
    StructField("city", StringType(), True)
])

df = spark.createDataFrame(data, schema=schema)

### Narrow Transformation


In [0]:
df = df.filter(col('city')=='New York')

In [0]:
display(df)

In [0]:
df.explain()

## Wide Transformation


In [0]:
display(df)


In [0]:
#df = df.groupBy("city").agg(max("age").alias("max_age"))
display(df)


In [0]:
df = df.groupby('city').agg(max(col('age')).alias("max_age"))
display(df)


## Repartition Vs Coalesce

In [0]:
#df.rdd.getNumPartitions()


In [0]:
#df.explain(True)
df.explain()


In [0]:
df=df.repartition(4)
#df.explain(True)

## Coalesce

In [0]:
df = df.coalesce(1)

In [0]:
df.explain()

# he data is read from a table scan (LocalTableScan) with columns age and city.
# Spark first computes a partial aggregation:
# → max(age) for each city (done locally).
# Then it shuffles data across the cluster based on city (hash partitioning).
# Next it finishes the aggregation (final max per city).
# The results are sorted by city and max_age.
# Data is repartitioned (4 partitions) using round-robin.
# Finally, everything is collected into 1 partition (Coalesce 1) for output.
# The whole query runs using Photon (Databricks’ optimized execution engine) and Adaptive Query Execution (AQE), meaning Spark can adjust the plan at runtime.

# One-line summary:
# It computes the maximum age per city, shuffles and aggregates data across the cluster, sorts the results, and returns a single final output.